In [1]:
# Repository Document Number Synchronization

import pathlib
import re
from collections import defaultdict

root = pathlib.Path('f:/projects/SmartCorePlatform')
md_files = sorted([p for p in root.rglob('*.md') if p.is_file()])
print(f'Found {len(md_files)} markdown files.')

basename_to_path = {p.name: p for p in md_files}
suffix_to_basenames = defaultdict(list)
for base in basename_to_path:
    m = re.match(r'^(\d{3})_(.+)$', base)
    if m:
        suffix_to_basenames[m.group(2)].append(base)

duplicates = {suffix: bases for suffix, bases in suffix_to_basenames.items() if len(bases) > 1}
print('Duplicate suffixes detected:', len(duplicates))
for suffix, bases in list(duplicates.items())[:10]:
    print(' ', suffix, bases)

ref_pattern = re.compile(r'(?P<path>(?:[A-Za-z0-9_./%&@()+\- ]*/)?)(?P<base>(?P<num>\d{3})_(?P<suffix>[^\s\)\]\"\']+?\.md))')
header_pattern = re.compile(r'^(?P<leading>#+\s*)(?P<base>\d{3}_[^\s]+\.md)(?P<rest>.*)$', re.MULTILINE)
docid_pattern = re.compile(r'^(?P<leading>\s*Document ID:\s*)(?P<num>\d{2,3})(?P<rest>\s*)$', re.MULTILINE)

modified_files = []
modified_references = []
header_mismatches = []
docid_mismatches = []
unresolved_references = []

for p in md_files:
    text = p.read_text(encoding='utf-8')
    original = text
    basename = p.name
    own_num = basename[:3] if len(basename) >= 3 and basename[:3].isdigit() else None
    own_suffix = basename[4:] if own_num else None

    def header_repl(m):
        base = m.group('base')
        if own_suffix and base.endswith(own_suffix) and base != basename:
            header_mismatches.append((p, base, basename))
            return m.group('leading') + basename + m.group('rest')
        return m.group(0)
    text = header_pattern.sub(header_repl, text)

    def docid_repl(m):
        if own_num and m.group('num') != own_num:
            docid_mismatches.append((p, m.group(0), own_num))
            return m.group('leading') + own_num + m.group('rest')
        return m.group(0)
    text = docid_pattern.sub(docid_repl, text)

    def ref_repl(m):
        path = m.group('path')
        base = m.group('base')
        suffix = m.group('suffix')
        if base in basename_to_path:
            return m.group(0)
        candidates = suffix_to_basenames.get(suffix, [])
        if len(candidates) == 1:
            replacement = candidates[0]
            if replacement != base:
                modified_references.append((p, base, replacement))
                return path + replacement
        elif len(candidates) > 1:
            unresolved_references.append((p, base, candidates))
        else:
            unresolved_references.append((p, base, None))
        return m.group(0)

    text = ref_pattern.sub(ref_repl, text)

    if text != original:
        p.write_text(text, encoding='utf-8')
        modified_files.append(p)

print('Completed normalization pass.')

still_unresolved = []
for p in md_files:
    text = p.read_text(encoding='utf-8')
    for m in ref_pattern.finditer(text):
        base = m.group('base')
        if base not in basename_to_path:
            still_unresolved.append((p, base))

report = {
    'modified_files': sorted({str(p.relative_to(root)).replace('\\','/') for p in modified_files}),
    'modified_references': [(str(p.relative_to(root)).replace('\\','/'), old, new) for p, old, new in modified_references],
    'header_mismatches': [(str(p.relative_to(root)).replace('\\','/'), old, new) for p, old, new in header_mismatches],
    'docid_mismatches': [(str(p.relative_to(root)).replace('\\','/'), old, new) for p, old, new in docid_mismatches],
    'initial_unresolved': [(str(p.relative_to(root)).replace('\\','/'), base, candidates) for p, base, candidates in unresolved_references],
    'still_unresolved': sorted({(str(p.relative_to(root)).replace('\\','/'), base) for p, base in still_unresolved}),
}

print('FILES_MODIFIED:', len(report['modified_files']))
for f in report['modified_files']:
    print('FILE', f)
print('REFERENCES_FIXED:', len(report['modified_references']))
for f, old, new in report['modified_references']:
    print('REF', f, old, '->', new)
print('HEADER_MISMATCHES_FIXED:', len(report['header_mismatches']))
for f, old, new in report['header_mismatches']:
    print('HEADER', f, old, '->', new)
print('DOCID_MISMATCHES_FIXED:', len(report['docid_mismatches']))
for f, old, new in report['docid_mismatches']:
    print('DOCID', f, old, '->', new)
print('INITIAL_UNRESOLVED:', len(report['initial_unresolved']))
for f, base, candidates in report['initial_unresolved']:
    print('UNRESOLVED', f, base, 'candidates:', candidates)
print('STILL_UNRESOLVED:', len(report['still_unresolved']))
for f, base in report['still_unresolved']:
    print('STILL_UNRESOLVED', f, base)


Found 95 markdown files.
Duplicate suffixes detected: 1
  SmartCore_Event_Model.md ['026_SmartCore_Event_Model.md', '035_SmartCore_Event_Model.md']
Completed normalization pass.
FILES_MODIFIED: 1
FILE SmartCore_Platform_Docs_v1/026_SmartCore_Event_Model.md
REFERENCES_FIXED: 0
HEADER_MISMATCHES_FIXED: 1
HEADER SmartCore_Platform_Docs_v1/026_SmartCore_Event_Model.md 035_SmartCore_Event_Model.md -> 026_SmartCore_Event_Model.md
DOCID_MISMATCHES_FIXED: 0
INITIAL_UNRESOLVED: 73
UNRESOLVED CONTEXT5.md 006_Implementation_Report.md candidates: None
UNRESOLVED SmartCore_Platform_Docs_v1/031_SmartCore_Core_Vocabulary.md 005_SmartCore_Semantic_Grammar.md candidates: None
UNRESOLVED SmartCore_Platform_Docs_v1/033_SmartCore_Execution_Boundary_Model.md 047_SmartCore_Architecture&%20Taxonomy_Layer_Model.md candidates: None
UNRESOLVED SmartCore_Platform_Docs_v1/047_SmartCore_Reference_Architecture.md 047_SmartCore_Architecture&%20Taxonomy_Layer_Model.md candidates: None
UNRESOLVED SmartCore_Platform_Do

In [3]:
import pathlib
import re
from collections import defaultdict, Counter, deque

root = pathlib.Path('f:/projects/SmartCorePlatform')
md_files = sorted([p for p in root.rglob('*.md') if p.is_file()])

basename_to_path = {p.name: p for p in md_files}

link_pattern = re.compile(r'\[([^\]]+)\]\(([^\)]+\.md)\)')
bare_ref_pattern = re.compile(r'(?<!\[)(?<!\()(?P<ref>\d{3}_[^\s\)\]\"\']+?\.md)')

def normalize_ref(ref):
    return ref.split('/')[-1].split('?')[0].split('#')[0]

def extract_internal_id(text, basename):
    m = re.search(r'^\s*Document ID:\s*(\d{2,3})\s*$', text, re.MULTILINE)
    if m:
        return m.group(1)
    m = re.match(r'^\s*#\s*(\d{3})_[^\r\n]+\.md', text)
    if m:
        return m.group(1)
    return ''

def extract_internal_title(text, basename):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return ''
    first = lines[0]
    if re.match(r'^#\s*\d{3}_[^\s]+\.md', first):
        return first.lstrip('#').strip().rsplit('.md', 1)[0]
    if first.startswith('#'):
        return first.lstrip('#').strip()
    return first

def extract_field(text, field):
    m = re.search(r'^\s*' + re.escape(field) + r':\s*(.+?)\s*$', text, re.MULTILINE)
    return m.group(1).strip() if m else ''

def extract_section(text, headings):
    lines = text.splitlines()
    section = []
    capture = False
    level = None
    for line in lines:
        h = re.match(r'^(#{1,6})\s*(.+)$', line)
        if h:
            title = h.group(2).strip().lower()
            if any(term in title for term in headings):
                capture = True
                level = len(h.group(1))
                continue
            if capture and len(h.group(1)) <= level:
                break
        elif capture:
            section.append(line)
    if not section and 'purpose' in headings:
        for i, line in enumerate(lines):
            if re.match(r'^#{1,6}\s*\d+\.\s*Purpose', line, re.IGNORECASE):
                for j in range(i+1, len(lines)):
                    if re.match(r'^#{1,6}\s*', lines[j]):
                        break
                    section.append(lines[j])
                break
    return '\n'.join([l.strip() for l in section if l.strip()])

def extract_document_references(text):
    refs = []
    for m in link_pattern.finditer(text):
        ref = normalize_ref(m.group(2))
        refs.append(ref)
    for m in bare_ref_pattern.finditer(text):
        ref = normalize_ref(m.group('ref'))
        refs.append(ref)
    return refs

records = []
for p in md_files:
    content = p.read_text(encoding='utf-8')
    filename = p.name
    doc_num = filename[:3] if re.match(r'^\d{3}_', filename) else ''
    internal_id = extract_internal_id(content, filename)
    internal_title = extract_internal_title(content, filename)
    version = extract_field(content, 'Version')
    status = extract_field(content, 'Status')
    purpose = extract_section(content, ['purpose'])
    dependencies = extract_section(content, ['depends on', 'dependencies', 'dependency'])
    authority = extract_section(content, ['authority', 'governance', 'decision', 'approval'])
    references = extract_document_references(content)
    records.append({
        'path': str(p.relative_to(root)).replace('\\', '/'),
        'filename': filename,
        'doc_number': doc_num,
        'internal_id': internal_id,
        'internal_title': internal_title,
        'version': version,
        'status': status,
        'purpose': purpose,
        'dependencies': dependencies,
        'authority': authority,
        'references': references,
    })

missing_refs = []
for r in records:
    for ref in r['references']:
        if ref not in basename_to_path:
            missing_refs.append((r['filename'], ref))

category_keywords = [
    ('Foundation', ['vision', 'foundational', 'overview', 'platform overview']),
    ('Meta Model', ['meta model', 'modeling rules', 'composition rules', 'domain modeling rules', 'core vocabulary', 'metal model']),
    ('Semantic Model', ['semantic', 'glossary', 'domain layer', 'event model', 'relation model', 'rule model', 'state model', 'time model', 'execution model', 'runtime model', 'vocabulary']),
    ('Architecture', ['architecture', 'reference architecture', 'architecture taxonomy', 'platform taxonomy']),
    ('Governance', ['governance', 'decision model', 'freeze', 'adr', 'module standards', 'governance model']),
    ('Runtime', ['runtime', 'execution boundary', 'codebase architecture', 'repository and package strategy', 'identity platform', 'core engine', 'module interaction', 'deployment', 'execution']),
    ('Infrastructure', ['integration', 'platform development guideline', 'codebase architecture', 'repository', 'package strategy']),
    ('Deployment', ['deployment', 'scaling', 'production']),
    ('Security', ['security', 'permission', 'identity and session', 'identity platform', 'tenancy', 'ownership']),
    ('Validation', ['validation', 'matrix', 'validator', 'test', 'consistency', 'audit', 'compliance']),
    ('AI Pipeline', ['ai', 'code generation', 'generation', 'blueprint validator']),
    ('Blueprint', ['blueprint', 'validator', 'build', 'package', 'developer experience']),
    ('Capability Model', ['capability', 'platform taxonomy', 'module standards', 'phase2 platform roadmap']),
    ('Reference', ['reference', 'cross reference', 'architecture consistency report', 'crossreference', 'audit report']),
    ('Glossary', ['glossary', 'vocabulary']),
    ('Standards', ['standards', 'module standards', 'blueprint standard']),
]
for r in records:
    key = r['filename'].lower().replace('_', ' ').replace('.md', '')
    title = r['internal_title'].lower()
    found = 'Other'
    for cat, kws in category_keywords:
        if any(kw in key or kw in title or kw in r['purpose'].lower() for kw in kws):
            found = cat
            break
    r['category'] = found
    summary = r['purpose'].replace('\n', ' ').strip() or r['internal_title']
    r['summary'] = (summary[:200] + '...') if len(summary) > 200 else summary
    r['responsibility'] = ('Define ' + r['category'] + ' concerns for SmartCore') if r['category'] != 'Other' else 'Document repository content or supporting material'

depends_on = {r['filename']: [normalize_ref(ref) for ref in r['references'] if normalize_ref(ref) in basename_to_path] for r in records}
referenced_by = defaultdict(list)
for src, targets in depends_on.items():
    for target in targets:
        referenced_by[target].append(src)

cycles = []
visited = {}
stack = []

def dfs(node):
    visited[node] = 1
    stack.append(node)
    for nb in depends_on.get(node, []):
        if visited.get(nb) == 1:
            cycle = stack[stack.index(nb):] + [nb]
            cycles.append(cycle)
        elif visited.get(nb) == 0:
            dfs(nb)
    visited[node] = 2
    stack.pop()
for r in records:
    visited[r['filename']] = 0
for r in records:
    if visited[r['filename']] == 0:
        dfs(r['filename'])

numbers = sorted([int(r['doc_number']) for r in records if r['doc_number']])
gaps = [(numbers[i], numbers[i+1]) for i in range(len(numbers)-1) if numbers[i+1] != numbers[i] + 1]
duplicate_titles = [title for title, count in Counter([r['internal_title'] for r in records]).items() if count > 1 and title]
filename_id_mismatches = [r for r in records if r['doc_number'] and r['internal_id'] and r['doc_number'] != r['internal_id']]

inventory_lines = ['# SmartCore Documentation Inventory Report', '']
inventory_lines.append('## Phase 1 — Repository Inventory')
inventory_lines.append('')
inventory_lines.append('| File | Internal ID | Title | Version | Status | Category |')
inventory_lines.append('| --- | --- | --- | --- | --- | --- |')
for r in sorted(records, key=lambda x: (x['doc_number'] or '999', x['filename'])):
    inventory_lines.append(f"| {r['filename']} | {r['internal_id']} | {r['internal_title']} | {r['version']} | {r['status']} | {r['category']} |")

inventory_lines.append('')
inventory_lines.append('## Phase 2 — Functional Classification')
inventory_lines.append('')
for r in sorted(records, key=lambda x: (x['category'], x['filename'])):
    inventory_lines.append(f"### {r['filename']} — {r['category']}")
    inventory_lines.append(f"- Primary responsibility: {r['responsibility']}")
    inventory_lines.append(f"- Summary: {r['summary']}")
    inventory_lines.append('')

inventory_lines.append('## Phase 3 — Dependency Graph')
inventory_lines.append('')
inventory_lines.append('### Document Dependencies')
inventory_lines.append('')
for r in sorted(records, key=lambda x: x['filename']):
    inventory_lines.append(f"#### {r['filename']}")
    inventory_lines.append(f"- Depends On: {', '.join(depends_on[r['filename']]) or 'None'}")
    inventory_lines.append(f"- Referenced By: {', '.join(sorted(referenced_by[r['filename']])) or 'None'}")
    inventory_lines.append('')

inventory_lines.append('### Most referenced documents')
inventory_lines.append('')
for target, callers in Counter({k: len(v) for k, v in referenced_by.items()}).most_common(20):
    inventory_lines.append(f"- {target}: {callers} inbound references")
inventory_lines.append('')
no_inbound = [r['filename'] for r in records if len(referenced_by[r['filename']]) == 0]
no_outbound = [r['filename'] for r in records if len(depends_on[r['filename']]) == 0]
inventory_lines.append('### Documents with no inbound references')
inventory_lines.append('')
inventory_lines.extend(f"- {f}" for f in sorted(no_inbound))
inventory_lines.append('')
inventory_lines.append('### Documents with no outbound references')
inventory_lines.append('')
inventory_lines.extend(f"- {f}" for f in sorted(no_outbound))
inventory_lines.append('')

inventory_lines.append('## Phase 4 — Consistency Analysis')
inventory_lines.append('')
inventory_lines.append(f"- Filename number != internal ID: {len(filename_id_mismatches)}")
for r in filename_id_mismatches[:20]:
    inventory_lines.append(f"  - {r['filename']} internal ID {r['internal_id']}")
inventory_lines.append('')
inventory_lines.append(f"- Duplicate titles: {len(duplicate_titles)}")
for title in duplicate_titles[:20]:
    inventory_lines.append(f"  - {title}")
inventory_lines.append('')
inventory_lines.append(f"- Missing referenced documents: {len(missing_refs)}")
for src, ref in missing_refs[:40]:
    inventory_lines.append(f"  - {src} -> {ref}")
inventory_lines.append('')
inventory_lines.append(f"- Numbering anomalies / gaps: {len(gaps)}")
for a, b in gaps[:20]:
    inventory_lines.append(f"  - gap between {a:03d} and {b:03d}")
inventory_lines.append('')
inventory_lines.append(f"- Circular dependency cycles detected: {len(cycles)}")
for cycle in cycles[:20]:
    inventory_lines.append(f"  - {' -> '.join(cycle)}")
inventory_lines.append('')

inventory_lines.append('## Phase 5 — Architecture Map')
inventory_lines.append('')
inventory_lines.append('SmartCore is a modular platform architecture combining formal semantic modeling, layered platform architecture, governance-driven evolution, and runtime execution boundaries. It centers on a core semantic model (SFMM), a platform taxonomy, module standards, and a governance / freeze policy that enforces consistency and evolution discipline.')
inventory_lines.append('')
inventory_lines.append('### Core conceptual layers')
inventory_lines.append('')
inventory_lines.append('- Foundation: vision, principles, and high-level platform overview.')
inventory_lines.append('- Meta Model: the formal semantics, modeling rules, and composition rules that define the SmartCore grammar.')
inventory_lines.append('- Semantic Model: domain vocabulary, event, relation, rule, state, and time models that define meaning and behavior.')
inventory_lines.append('- Architecture: reference architecture, taxonomy, platform structure, and module standards.')
inventory_lines.append('- Governance: decision models, freeze policy, and documentation governance that manage evolution.')
inventory_lines.append('- Runtime: execution boundaries, runtime model, codebase architecture, and repository/package strategy.')
inventory_lines.append('- Blueprint and AI Pipeline: blueprint standards, validator specification, and AI code generation requirements.')
inventory_lines.append('')
inventory_lines.append('### Major subsystems')
inventory_lines.append('')
inventory_lines.append('- Semantic foundation and vocabulary standardization')
inventory_lines.append('- Architectural taxonomy and module governance')
inventory_lines.append('- Runtime execution models and implementation architecture')
inventory_lines.append('- Blueprint-driven development and AI generation support')
inventory_lines.append('')
inventory_lines.append('### Governance model')
inventory_lines.append('')
inventory_lines.append('- Canonical governance is defined in the governance and decision model, module standards, and freeze policy documents.')
inventory_lines.append('- Documentation governance is reinforced by architecture consistency and audit reports.')
inventory_lines.append('')
inventory_lines.append('### Runtime model')
inventory_lines.append('')
inventory_lines.append('- Runtime is anchored by execution boundary and runtime model documents, with codebase and repository architecture providing implementation guidance.')
inventory_lines.append('')
inventory_lines.append('### Blueprint model')
inventory_lines.append('')
inventory_lines.append('- Blueprint standards, validator specification, and MVP guidance define the blueprint lifecycle and enforcement model.')
inventory_lines.append('')
inventory_lines.append('### AI Generation model')
inventory_lines.append('')
inventory_lines.append('- AI generation is represented by the SmartCore AI code generation specification, which outlines how AI artifacts integrate with the platform documentation and blueprint pipeline.')
inventory_lines.append('')
inventory_lines.append('### Canonical documents')
inventory_lines.append('')
inventory_lines.append('- Architecture: 047_SmartCore_Reference_Architecture.md, 048_SmartCore_Architecture& Taxonomy_Layer_Model.md, 049_SmartCore_Platform_Taxonomy.md')
inventory_lines.append('- Governance: 051_SmartCore_Governance_and_Decision_Model.md, 050_SmartCore_Module_Standards.md')
inventory_lines.append('- Runtime: 032_SmartCore_Execution_Boundary_Model.md, 033_SmartCore_Runtime_Model.md, 060_SmartCore_Codebase_Architecture.md, 061_SmartCore_Repository_and_Package_Strategy.md')
inventory_lines.append('- Blueprint: 064_SmartCore_Blueprint_Standard.md, 065_SmartCore_Blueprint_Validator_Specification.md')
inventory_lines.append('- AI Pipeline: 066_SmartCore_AI_Code_Generation_Specification.md')
inventory_lines.append('')

inventory_lines.append('## Phase 6 — Executive Summary')
inventory_lines.append('')
inventory_lines.append('### Repository statistics')
inventory_lines.append('')
category_counts = Counter(r['category'] for r in records)
inventory_lines.append(f"- Total documents: {len(records)}")
inventory_lines.append(f"- Documents by category:")
for cat, count in category_counts.most_common():
    inventory_lines.append(f"  - {cat}: {count}")
inventory_lines.append(f"- Total referenced document links: {sum(len(r['references']) for r in records)}")
inventory_lines.append(f"- Broken or missing references: {len(missing_refs)}")
inventory_lines.append('')
inventory_lines.append('### Top 20 most important documents')
inventory_lines.append('')
important = Counter({r['filename']: len(referenced_by[r['filename']]) for r in records})
for filename, count in important.most_common(20):
    inventory_lines.append(f"- {filename}: {count} inbound references")
inventory_lines.append('')
inventory_lines.append('### Top findings')
inventory_lines.append('')
if missing_refs:
    inventory_lines.append('**Critical**: Missing referenced documents and broken links are present in the repository. These should be reviewed before any publication or formal freeze.')
if filename_id_mismatches:
    inventory_lines.append('**Major**: Some documents have inconsistent filename numbers versus internal document IDs.')
if cycles:
    inventory_lines.append('**Major**: Circular dependencies were detected in the document reference graph.')
if duplicate_titles:
    inventory_lines.append('**Medium**: Duplicate document titles may indicate overlapping responsibilities or content duplication.')
if gaps:
    inventory_lines.append('**Medium**: Numbering gaps exist, which may reflect renumbering drift or missing historical documents.')
inventory_lines.append('**Minor**: A single duplicate suffix was detected for SmartCore_Event_Model.md across two filenames.')
inventory_lines.append('')

map_lines = ['# SmartCore Documentation Dependency Map', '']
map_lines.append('## Document Dependency Summary')
map_lines.append('')
for r in sorted(records, key=lambda x: x['filename']):
    map_lines.append(f"### {r['filename']}")
    map_lines.append(f"- Depends On: {', '.join(depends_on[r['filename']]) or 'None'}")
    map_lines.append(f"- Referenced By: {', '.join(sorted(referenced_by[r['filename']])) or 'None'}")
    map_lines.append('')
map_lines.append('## Most Referenced Documents')
for target, count in important.most_common(40):
    map_lines.append(f"- {target}: {count}")
map_lines.append('')
map_lines.append('## Documents with no inbound references')
for f in sorted(no_inbound):
    map_lines.append(f"- {f}")
map_lines.append('')
map_lines.append('## Documents with no outbound references')
for f in sorted(no_outbound):
    map_lines.append(f"- {f}")
map_lines.append('')
map_lines.append('## Broken or Missing References')
for src, ref in missing_refs:
    map_lines.append(f"- {src} -> {ref}")
map_lines.append('')
map_lines.append('## Circular Dependency Cycles')
for cycle in cycles:
    map_lines.append(f"- {' -> '.join(cycle)}")
map_lines.append('')

print('\n'.join(inventory_lines[:40]))
print('...')
print(len(inventory_lines), 'lines in inventory report')
print('\n'.join(map_lines[:40]))
print('...')
print(len(map_lines), 'lines in dependency map')


# SmartCore Documentation Inventory Report

## Phase 1 — Repository Inventory

| File | Internal ID | Title | Version | Status | Category |
| --- | --- | --- | --- | --- | --- |
| 000_SmartCore_Vision.md |  | SmartCore Vision | 0.1 (Draft) |  | Foundation |
| 001_SmartCore_Foundational_Principles.md |  | SmartCore Foundational Principles | 1.0 |  | Foundation |
| 002_SmartCore_Meta_Model.md |  | SmartCore Meta Model | 1.0 |  | Meta Model |
| 003_SmartCore_Modeling_Rules.md |  | SmartCore Modeling Rules | 1.0 |  | Meta Model |
| 004_SmartCore_Composition_Rules.md |  | SmartCore Composition Rules | 1.1 |  | Meta Model |
| 005_SmartCore_Domain_Layer.md |  | SmartCore Domain Layer | 1.0 |  | Semantic Model |
| 006_SmartCore_Execution_Model.md |  | SmartCore Execution Model | 1.0 |  | Semantic Model |
| 007_SmartCore_Economic_Model.md |  | SmartCore Economic Model | 1.0 |  | Security |
| 008_Validation Matrix.md |  | SFMM-08 — Validation Matrix (Core Concept Test System) |  |  | Semantic Mo

In [5]:
import tempfile
from pathlib import Path

temp_dir = Path(tempfile.gettempdir())
audit_path = temp_dir / 'SmartCore_Audit_Report.md'
dep_map_path = temp_dir / 'SmartCore_Dependency_Map.md'

audit_path.write_text('\n'.join(inventory_lines), encoding='utf-8')
dep_map_path.write_text('\n'.join(map_lines), encoding='utf-8')
print('Audit path:', audit_path)
print('Dependency map path:', dep_map_path)


Audit path: C:\Users\Amir\AppData\Local\Temp\SmartCore_Audit_Report.md
Dependency map path: C:\Users\Amir\AppData\Local\Temp\SmartCore_Dependency_Map.md
